In [11]:
%pip install ipykernel pandas matplotlib playwright pydantic-settings httpx -q -U

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import glob
import os
from datetime import datetime
import hashlib
import re


# Configurações de exibição do Pandas (DX - Developer Experience)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:.2f}'.format)


In [93]:
# Dicionário de tipos para garantir integridade na carga (Fail-fast)
# Colunas com DtypeWarning são forçadas para 'str' aqui
dtypes_carga = {
    'Protocolo': str,
    'Teleconsulta': str,
    'Origem da Regulação': str,
    'Situação': str,
    'Complexidade': str,
    'Risco Cor': str,
    'Cartão SUS': str, # Recomendado ler como str para evitar perda de zeros à esquerda
    'CPF': str,
    'CEP': str
}

# Colunas de data para o parse_dates
colunas_data = [
    "Data Solicitação", 
    "Data de Nascimento", 
    "Data do Cadastro"
]

dfs = []
arquivos_csv = glob.glob("dados_gercon_*.csv") or (["dados_gercon.csv"] if os.path.exists("dados_gercon.csv") else [])

for arquivo in arquivos_csv:
    try:
        # Injeção de tipos e parse de datas direto na origem
        df_temp = pd.read_csv(
            arquivo, 
            encoding='utf-8', 
            quoting=1, 
            dtype=dtypes_carga,
            parse_dates=[c for c in colunas_data if c in dtypes_carga or True], # Tenta parsear se a coluna existir
            dayfirst=True,
            low_memory=False
        )
        
        # Metadados para Observabilidade
        df_temp['source_file'] = arquivo
        df_temp['ingestion_at'] = datetime.now()
        
        dfs.append(df_temp)
    except Exception as e:
        print(f"❌ Erro ao ler {arquivo}: {e}")

# Consolidação Global
df = pd.concat(dfs, ignore_index=True, sort=False)

# 3. Otimização Pós-Carga (Memory Management)
# Agora convertemos para 'category' as colunas de baixa cardinalidade
for col in ['Situação', 'Complexidade', 'Risco Cor']:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Limpeza baseada na Ubiquitous Language (Protocolo válido deve conter '-')
if 'Protocolo' in df.columns:
    df = df[df['Protocolo'].str.contains('-', na=False)]
    df = df.drop_duplicates(subset=['Protocolo'], keep='last')

print(f"Total de registros únicos: {len(df)}")

# Mascaramento
def anonymize(value, salt="secret_key"):
    if pd.isna(value): return None
    return hashlib.sha256(f"{value}{salt}".encode()).hexdigest()

# Lista para remoção imediata ou Hashing (Identificadores Diretos)
identificadores_diretos = [
    "Nome do Paciente", "CPF", "Cartão SUS", "Nome da Mãe", 
]

# Lista para Anonimização/Generalização (Saúde e Sensíveis)
sensiveis_saude = [
    "Cor", "Sexo", "CID Código", "CID Descrição", 
    "Histórico Quadro Clínico", "Complexidade", "Risco Cor"
]

# Lista para Mascaramento (Localização e Datas)
quase_identificadores = [
    "Data de Nascimento", "CEP", "Logradouro", "Número", 
    "Complemento", "Bairro", "Município de Residência", 
    "Data Solicitação", "Data do Cadastro"
]

for col in identificadores_diretos:
    df[col] = df[col].apply(anonymize)


Total de registros únicos: 123211


In [94]:
pattern = r'\[(?P<Data_Evolucao>.*?)\s*\|\s*(?P<Tipo_Informacao>.*?)\s*\|\s*(?P<Origem_Informacao>.*?)\]:\s*(?P<Texto_Evolucao>.*)'

def explode_historico(df_input):
    df_result = df_input.copy()
    
    # 1. Split robusto: O separador ' | ' entre evoluções foi definido no master_scraper.py.
    # IMPORTANTE: Usamos regex no split para não quebrar dentro do colchete.
    df_result['Histórico Quadro Clínico'] = df_result['Histórico Quadro Clínico'].str.split(r'\s*\|\s*(?=\n\n\[)')
    df_result = df_result.explode('Histórico Quadro Clínico')
    
    # 2. Limpeza de ruído (Remoção dos \n\n iniciais inseridos pelo scraper)
    df_result['Histórico Quadro Clínico'] = df_result['Histórico Quadro Clínico'].str.strip()
    
    # 3. Extração com Regex considerando múltiplas linhas no texto (re.DOTALL)
    detalhes = df_result['Histórico Quadro Clínico'].str.extract(pattern, flags=re.DOTALL)
    
    # 4. Join dos dados extraídos
    df_final = pd.concat([df_result.drop(columns=['Histórico Quadro Clínico']), detalhes], axis=1)
    
    # 5. Tipagem e Sanidade (Robust)
    df_final['Data_Evolucao'] = pd.to_datetime(df_final['Data_Evolucao'], dayfirst=True, errors='coerce')
    
    return df_final

# Execução da extração corrigida
df_temp = df.copy()
df = explode_historico(df_temp)

In [95]:
paciente = '1a477dddf997afcf1a8c8630c694d6fdbe267d4a440a3dae1386b9d30c3a195f'

mask = df['Nome do Paciente'] == paciente

df[mask]

,Protocolo,Situação,Origem da Lista,Data Solicitação,Nome do Paciente,Data de Nascimento,Sexo,Cor,CPF,Nome da Mãe,Cartão SUS,Logradouro,Número,Complemento,Bairro,CEP,Município de Residência,Nacionalidade,Ordem Judicial,Especialidade Mãe,Especialidade,Especialidade Descrição Auxiliar,Especialidade CBO,Tipo de Regulação,Teleconsulta,Status da Especialidade,Complexidade,Risco Cor,Cor Regulador,Pontos Gravidade,Pontos Tempo,Pontuação,Situação Final,CID Código,CID Descrição,Unidade Solicitante,Operador,Usuário Solicitante,Unidade Razão Social,Unidade Descrição,Central de Regulação,Origem da Regulação,Data do Cadastro,Médico Solicitante,source_file,ingestion_at,Data_Evolucao,Tipo_Informacao,Origem_Informacao,Texto_Evolucao
109327,16-06-0000590-8,REALIZADA,Outras,2016-06-24 16:51:00,1a477dddf997afcf1a8c8630c694d6fdbe267d4a440a3dae1386b9d30c3a195f,1966-07-28,Masculino,NaN,8515f2e1927e46bb7ab9b928ba72fe846f2f0853c5dc8cd86ccd195194bc3402,811fd5b18de5def612dbee5641bbf778294b472b23d80200d6d68035af257617,5742aa6d4f80649b12212ea5102645337fc920a1b3a629601dd7b23be2c2523f,TUPI,29,NaN,SALOME,94828710,ALVORADA,Brasileiro,,ONCOLOGIA,ONCOLOGIA CIRURGIA DE CABECA E PESCOCO,NaN,MEDICO ONCOLOGISTA CLINICO,REGULADA,False,True,ALTA,VERMELHO,AMARELO,1000,0,1000,REALIZADA,C028,NEOPLASIA MALIGNA DA LÍNGUA COM LESÃO INVASIVA,UBS CAMPOS VERDES ALVORADA,ELIDE MALLMANN BERLANDA TEIXEIRA,BARBARA CAPITANIO DE SOUZA,PREFEITURA MUNICIPAL DE ALVORADA,CENTRO DE SAUDE/UNIDADE BASICA,CENTRAL DE REGULACAO AMBULATORIAL DO RIO GRANDE DO SUL,NaN,2016-06-24 16:52:00,ELIDE MALLMANN BERLANDA TEIXEIRA,dados_gercon_outras (Copy).csv,2026-03-24 09:33:15.293705,2016-06-24 16:52:00,Descrição do quadro clínico,ELIDE MALLMANN BERLANDA TEIXEIRA,"leSAO NODULAR, ULCERADA, ENDURECIDA, +-5CM NO DORSO DE LINGUA ESQ, DOLORIDA, EVOLUÇÃO 30 DIAS (R..."
109327,16-06-0000590-8,REALIZADA,Outras,2016-06-24 16:51:00,1a477dddf997afcf1a8c8630c694d6fdbe267d4a440a3dae1386b9d30c3a195f,1966-07-28,Masculino,NaN,8515f2e1927e46bb7ab9b928ba72fe846f2f0853c5dc8cd86ccd195194bc3402,811fd5b18de5def612dbee5641bbf778294b472b23d80200d6d68035af257617,5742aa6d4f80649b12212ea5102645337fc920a1b3a629601dd7b23be2c2523f,TUPI,29,NaN,SALOME,94828710,ALVORADA,Brasileiro,,ONCOLOGIA,ONCOLOGIA CIRURGIA DE CABECA E PESCOCO,NaN,MEDICO ONCOLOGISTA CLINICO,REGULADA,False,True,ALTA,VERMELHO,AMARELO,1000,0,1000,REALIZADA,C028,NEOPLASIA MALIGNA DA LÍNGUA COM LESÃO INVASIVA,UBS CAMPOS VERDES ALVORADA,ELIDE MALLMANN BERLANDA TEIXEIRA,BARBARA CAPITANIO DE SOUZA,PREFEITURA MUNICIPAL DE ALVORADA,CENTRO DE SAUDE/UNIDADE BASICA,CENTRAL DE REGULACAO AMBULATORIAL DO RIO GRANDE DO SUL,NaN,2016-06-24 16:52:00,ELIDE MALLMANN BERLANDA TEIXEIRA,dados_gercon_outras (Copy).csv,2026-03-24 09:33:15.293705,2016-06-24 16:52:00,Enviado para regulação,ELIDE MALLMANN BERLANDA TEIXEIRA,Especialidade Regulada. Unidade não municipal.
109327,16-06-0000590-8,REALIZADA,Outras,2016-06-24 16:51:00,1a477dddf997afcf1a8c8630c694d6fdbe267d4a440a3dae1386b9d30c3a195f,1966-07-28,Masculino,NaN,8515f2e1927e46bb7ab9b928ba72fe846f2f0853c5dc8cd86ccd195194bc3402,811fd5b18de5def612dbee5641bbf778294b472b23d80200d6d68035af257617,5742aa6d4f80649b12212ea5102645337fc920a1b3a629601dd7b23be2c2523f,TUPI,29,NaN,SALOME,94828710,ALVORADA,Brasileiro,,ONCOLOGIA,ONCOLOGIA CIRURGIA DE CABECA E PESCOCO,NaN,MEDICO ONCOLOGISTA CLINICO,REGULADA,False,True,ALTA,VERMELHO,AMARELO,1000,0,1000,REALIZADA,C028,NEOPLASIA MALIGNA DA LÍNGUA COM LESÃO INVASIVA,UBS CAMPOS VERDES ALVORADA,ELIDE MALLMANN BERLANDA TEIXEIRA,BARBARA CAPITANIO DE SOUZA,PREFEITURA MUNICIPAL DE ALVORADA,CENTRO DE SAUDE/UNIDADE BASICA,CENTRAL DE REGULACAO AMBULATORIAL DO RIO GRANDE DO SUL,NaN,2016-06-24 16:52:00,ELIDE MALLMANN BERLANDA TEIXEIRA,dados_gercon_outras (Copy).csv,2026-03-24 09:33:15.293705,2016-07-07 16:04:00,Local,CRISTIANO OLIVEIRA DE SOUZA,"Irmandade Da Santa Casa De Misericordia De Porto Alegre, R Prof Annes Dias, 295, Sala: 3 - Hospi..."
109327,16-06-0000590-8,REALIZADA,Outras,2016-06-24 16:51:00,1a477dddf997afcf1a

In [97]:
df['Origem da Lista'].unique()

<StringArray>
['Outras', 'Pendentes', 'Expiradas', 'Fila de Espera',
 'Agendadas e Confirmadas']
Length: 5, dtype: str